# Phishing Detection with QLoRA

Fine-tune a small open-weight LLM (**Qwen2.5-0.5B-Instruct**) to classify messages as **phishing** or **legitimate**, using **4-bit QLoRA** (PEFT/LoRA) on a single GPU, then measure the **base model vs. the fine-tuned model** on a held-out test set.

**How to run (Google Colab):** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`. Takes roughly 15-25 minutes end to end.

> **Keep this browser tab open and in the foreground** while it runs. Free Colab disconnects runtimes that look idle, so switching away for several minutes can kill the session and you would have to start over.

## 1. Install dependencies

In [ ]:
!pip install -q -U "transformers>=4.46,<4.48" "peft>=0.13,<0.15" "datasets>=3.0,<4.0" "accelerate>=1.0" "bitsandbytes>=0.44" scikit-learn matplotlib
# torchvision is unused in this project, and Colab's preinstalled build clashes with
# `datasets` (ImportError: cannot import name 'VideoReader'). Remove it so the dataset
# tensorizing path can never trip on it.
!pip uninstall -y -q torchvision

## 2. Imports and configuration

In [ ]:
import collections, random
import numpy as np
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# A single T4 is enough for a 0.5B model in 4-bit.
assert torch.cuda.is_available(), "No GPU. In Colab: Runtime > Change runtime type > T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_LEN  = 512
LABELS   = {0: "legitimate", 1: "phishing"}

SYSTEM_PROMPT = (
    "You are a security classifier. Decide whether a message is a phishing "
    "attempt or legitimate. Answer with exactly one word: 'phishing' or 'legitimate'."
)

def user_prompt(text):
    text = (text or "").strip().replace("\n", " ")
    return (
        "Classify the following message.\n\n"
        "Message:\n" + text[:2000] + "\n\n"
        "Answer (phishing or legitimate):"
    )

## 3. Load and prepare the dataset

A public phishing-email dataset. We detect the text/label columns, map labels to a binary target (1 = phishing, 0 = legitimate), drop empties, then make a stratified train/test split.

> If the `load_dataset` line fails, the dataset id or columns may have changed. An alternative is `ealvaradob/phishing-dataset` (config `"texts"`).

In [ ]:
raw = load_dataset("zefang-liu/phishing-email-dataset", split="train")
print(raw)

cols = raw.column_names
text_col  = next((c for c in cols if c.lower() in ("email text","text","body","message","sms","content")), None)
label_col = next((c for c in cols if c.lower() in ("email type","label","type","class","target")), None)
assert text_col and label_col, f"Could not find text/label columns in {cols}"
print("Using text column:", repr(text_col), "| label column:", repr(label_col))

def to_binary(v):
    s = str(v).strip().lower()
    if "phish" in s or s in ("1", "1.0"): return 1
    if "safe" in s or "legit" in s or "ham" in s or s in ("0", "0.0"): return 0
    return 0

rows = []
for ex in raw:
    t = ex[text_col]
    if t is None:
        continue
    t = str(t).strip()
    if len(t) < 5:
        continue
    rows.append({"text": t, "label": to_binary(ex[label_col])})

data = Dataset.from_list(rows).shuffle(seed=SEED)
print("total usable rows:", len(data), "| class counts:", collections.Counter(data["label"]))

# Subsample for a fast, reproducible run.
data = data.select(range(min(len(data), 4000)))

# Stratified split with scikit-learn on row indices. Doing this through datasets'
# own stratify_by_column path tensorizes the labels, which on Colab reaches into
# torchvision and can ImportError; splitting on indices sidesteps that entirely.
from sklearn.model_selection import train_test_split as sk_train_test_split
labels = data["label"]
train_idx, test_idx = sk_train_test_split(
    list(range(len(data))), test_size=0.15, random_state=SEED, stratify=labels,
)
train_ds = data.select(train_idx)
test_ds  = data.select(test_idx)
test_ds  = test_ds.select(range(min(len(test_ds), 400)))  # cap test size to keep eval fast
print("train:", len(train_ds), "| test:", len(test_ds))

## 4. Load the tokenizer and render training examples

Each example becomes a chat with a system instruction, the message, and the correct label as the assistant's reply. We render it to text with the model's chat template, then tokenize.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def render(ex):
    msgs = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_prompt(ex["text"])},
        {"role": "assistant", "content": LABELS[ex["label"]]},
    ]
    return {"rendered": tok.apply_chat_template(msgs, tokenize=False)}

train_rendered = train_ds.map(render)
train_tok = train_rendered.map(
    lambda ex: tok(ex["rendered"], truncation=True, max_length=MAX_LEN),
    remove_columns=train_rendered.column_names,
)
print("Example training sequence:\n")
print(tok.decode(train_tok[0]["input_ids"]))

## 5. Load the base model in 4-bit (QLoRA)

In [ ]:
# float16 compute dtype: the T4 (compute capability 7.5) does not support bfloat16.
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
model.config.use_cache = True
print("Loaded", MODEL_ID, "in 4-bit.")

## 6. Baseline: evaluate the model *before* fine-tuning

We prompt the base model, read its one-word answer, and score it against the ground truth. This is our baseline, the number the fine-tune has to beat.

In [ ]:
from tqdm.auto import tqdm
tok.padding_side = "left"  # correct side for generation

@torch.no_grad()
def predict(m, text):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt(text)},
    ]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(m.device)
    out = m.generate(**inp, max_new_tokens=3, do_sample=False, pad_token_id=tok.pad_token_id)
    gen = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
    return 1 if "phish" in gen else 0

def evaluate(m, ds, name):
    y_true, y_pred = [], []
    for ex in tqdm(ds, desc=name):
        y_true.append(ex["label"]); y_pred.append(predict(m, ex["text"]))
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    print(f"\n{name}:  acc={acc:.3f}  precision={p:.3f}  recall={r:.3f}  f1={f1:.3f}")
    print("confusion matrix  [[TN FP] [FN TP]]:\n", confusion_matrix(y_true, y_pred))
    return {"name": name, "accuracy": acc, "precision": p, "recall": r, "f1": f1}

model.eval()
baseline = evaluate(model, test_ds, "base model")

## 7. Fine-tune with QLoRA

We attach LoRA adapters to the 4-bit model and train only those (a tiny fraction of the parameters), using a memory-efficient paged 8-bit optimizer.

In [ ]:
tok.padding_side = "right"      # correct side for training
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir="outputs",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    save_strategy="no",
    report_to="none",
    seed=SEED,
)
trainer = Trainer(
    model=model, args=args, train_dataset=train_tok,
    data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
)
trainer.train()

## 8. Evaluate the fine-tuned model on the same test set

In [ ]:
tok.padding_side = "left"
model.config.use_cache = True
model.eval()
finetuned = evaluate(model, test_ds, "fine-tuned (QLoRA)")

## 9. Compare base vs. fine-tuned

In [ ]:
import matplotlib.pyplot as plt

metrics  = ["accuracy", "precision", "recall", "f1"]
base_vals = [baseline[m] for m in metrics]
ft_vals   = [finetuned[m] for m in metrics]

print(f"{'metric':10s}  {'base':>7s}  {'fine-tuned':>10s}  {'delta':>7s}")
for m in metrics:
    print(f"{m:10s}  {baseline[m]:7.3f}  {finetuned[m]:10.3f}  {finetuned[m]-baseline[m]:+7.3f}")

x = np.arange(len(metrics)); w = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, base_vals, w, label="base", color="#94a3b8")
ax.bar(x + w/2, ft_vals, w, label="fine-tuned (QLoRA)", color="#16a34a")
ax.set_xticks(x); ax.set_xticklabels(metrics); ax.set_ylim(0, 1.05)
ax.set_title("Qwen2.5-0.5B: base vs. QLoRA fine-tuned (phishing detection)")
ax.legend()
for i, v in enumerate(base_vals): ax.text(i - w/2, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
for i, v in enumerate(ft_vals):   ax.text(i + w/2, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
plt.tight_layout(); plt.savefig("results.png", dpi=150); plt.show()

## 10. Save the LoRA adapter

In [ ]:
import shutil
model.save_pretrained("phishing-qlora-adapter")
tok.save_pretrained("phishing-qlora-adapter")
shutil.make_archive("phishing-qlora-adapter", "zip", "phishing-qlora-adapter")
print("Saved adapter to phishing-qlora-adapter/ and phishing-qlora-adapter.zip")

## 11. Results summary (copy this back)

Paste the block below so the README can be filled with the real numbers, and download `results.png`.

In [ ]:
print("=== COPY THIS BACK ===")
print("model:", MODEL_ID)
print("train_size:", len(train_ds), "test_size:", len(test_ds))
for m in metrics:
    print(f"{m}: base={baseline[m]:.3f} finetuned={finetuned[m]:.3f} delta={finetuned[m]-baseline[m]:+.3f}")
print("base_confusion / ft_confusion printed above in cells 6 and 8")